## **Prepocessing**

Import Library

In [ ]:
# 1. Import library
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
import joblib
import re


Input Data

In [ ]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path ke file Excel (ganti sesuai lokasi file di Drive-mu)
file_path = "/content/drive/MyDrive/Semester 7/PPW/pta_all.csv"

# Baca file Excel
df = pd.read_csv(file_path)

# Tampilkan nama kolom
print("Nama-nama kolom:")
print(df.columns.tolist())

# Tampilkan 100 baris pertama
print("\nContoh 100 baris pertama:")
print(df.head(100))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Nama-nama kolom:
['id', 'penulis', 'judul', 'abstrak_id', 'abstrak_en', 'pembimbing_pertama', 'pembimbing_kedua', 'prodi']

Contoh 100 baris pertama:
             id                           penulis  \
0   80111100012               Dyah Ayu Citra Seza   
1   80111100002                  Maulina Nurlaily   
2   70111100060               Moh. Samsul Hidayat   
3   90111100077  TOMMY ADITYA PARLINDUNGAN MARBUN   
4   70111200007                RICA YENA IMADHORA   
..          ...                               ...   
95  90111100078             Pradana Anggara Murti   
96  80111100065                               NaN   
97  70111100030                           ERFANDI   
98  80111100024               RIDLO ERFIN SANTOSO   
99  70111200008                               NaN   

                                                judul  \
0   Implementasi Fungsi Leg

**Prepocessing**



In [ ]:
# Stopword lokal
stopword_indonesia = set([
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan", "karena",
    "ada", "saya", "kami", "kita", "mereka", "pada", "adalah", "juga", "tidak",
    "ya", "kok", "loh", "banget", "sih", "jadi", "udah", "lagi", "aja", "dong", "nih"
])

# Normalisasi kata
normalisasi_kata = {
    "bangeettt": "banget", "bgt": "banget", "bgtt": "banget",
    "gk": "tidak", "ga": "tidak", "nggak": "tidak",
    "dr": "dari", "tp": "tapi", "tdk": "tidak",
    "sy": "saya", "lg": "lagi"
}

def normalize_kata(tokens):
    return [normalisasi_kata.get(word, word) for word in tokens]

def simple_tokenize(text):
    return text.split()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = simple_tokenize(text)
    tokens = normalize_kata(tokens)
    tokens = [word for word in tokens if word not in stopword_indonesia and len(word) > 2]
    return " ".join(tokens)

# Terapkan preprocessing
df['preprocessed'] = df['abstrak_id'].astype(str).apply(preprocess)

In [ ]:
import re
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Buat stemmer Sastrawi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Ambil 5 data awal
sample_data = df['abstrak_id'].astype(str).head(5)

# List untuk simpan hasil
stopwords_removed = []
cleaning = []
normalized = []
stemming = []
tokenizing = []

for text in sample_data:
    # Tokenisasi awal (pisah kata)
    tokens = text.split()

    # Stopword removal
    tokens_sw = [word for word in tokens if word not in stopword_indonesia and len(word) > 2]
    stopwords_removed.append(tokens_sw)

    # Cleaning (hapus karakter non-huruf)
    tokens_clean = [re.sub(r'[^a-zA-Z]', '', word) for word in tokens_sw if re.sub(r'[^a-zA-Z]', '', word) != ""]
    cleaning.append(tokens_clean)

    # Normalisasi (pembakuan kata)
    tokens_norm = [normalisasi_kata.get(word.lower(), word.lower()) for word in tokens_clean]
    normalized.append(tokens_norm)

    # Stemming
    tokens_stem = [stemmer.stem(word) for word in tokens_norm]
    stemming.append(tokens_stem)

    # Tokenisasi akhir
    tokenizing.append(tokens_stem)

# Simpan hasil ke DataFrame
tahapan_df = pd.DataFrame({
    'Asli': sample_data,
    'Stopword Removal': stopwords_removed,
    'Cleaning': cleaning,
    'Normalisasi (Ejaan Baku)': normalized,
    'Stemming': stemming,
    'Tokenisasi Akhir': tokenizing
})

# Tampilkan hasil
from IPython.display import display
display(tahapan_df)


,Asli,Stopword Removal,Cleaning,Normalisasi (Ejaan Baku),Stemming,Tokenisasi Akhir
0,ABSTRAK\r\n\r\n Implementasi Fungsi Legi...,"[ABSTRAK, Implementasi, Fungsi, Legislasi, DPR...","[ABSTRAK, Implementasi, Fungsi, Legislasi, DPR...","[abstrak, implementasi, fungsi, legislasi, dpr...","[abstrak, implementasi, fungsi, legislasi, dpr...","[abstrak, implementasi, fungsi, legislasi, dpr..."
1,Badan Usaha Milik Negara (BUMN) adalah Badan u...,"[Badan, Usaha, Milik, Negara, (BUMN), Badan, u...","[Badan, Usaha, Milik, Negara, BUMN, Badan, usa...","[badan, usaha, milik, negara, bumn, badan, usa...","[badan, usaha, milik, negara, bumn, badan, usa...","[badan, usaha, milik, negara, bumn, badan, usa..."
2,Kasus narkoba tidak henti-hentinya terdengar d...,"[Kasus, narkoba, henti-hentinya, terdengar, me...","[Kasus, narkoba, hentihentinya, terdengar, med...","[kasus, narkoba, hentihentinya, terdengar, med...","[kasus, narkoba, hentihentinya, dengar, media,...","[kasus, narkoba, hentihentinya, dengar, media,..."
3,Produk elektronik adalah suatu benda bergerak ...,"[Produk, elektronik, suatu, benda, bergerak, d...","[Produk, elektronik, suatu, benda, bergerak, d...","[produk, elektronik, suatu, benda, bergerak, d...","[produk, elektronik, suatu, benda, gerak, hasi...","[produk, elektronik, suatu, benda, gerak, hasi..."
4,nan,[nan],[nan],[nan],[nan],[nan]


In [ ]:
import re
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Buat stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Ambil semua data (ubah ke string dulu)
sample_data = df['abstrak_id'].astype(str)

# List untuk menyimpan hasil
stopwords_removed = []
cleaning = []
normalized = []
stemming = []
tokenizing = []

for text in sample_data:
    # Tokenisasi awal
    tokens = text.split()

    # Stopword removal
    tokens_sw = [word for word in tokens if word.lower() not in stopword_indonesia and len(word) > 2]
    stopwords_removed.append(tokens_sw)

    # Cleaning (hapus karakter non-huruf)
    tokens_clean = [re.sub(r'[^a-zA-Z]', '', word) for word in tokens_sw if re.sub(r'[^a-zA-Z]', '', word) != ""]
    cleaning.append(tokens_clean)

    # Normalisasi (pembakuan kata)
    tokens_norm = [normalisasi_kata.get(word.lower(), word.lower()) for word in tokens_clean]
    normalized.append(tokens_norm)

    # Stemming
    tokens_stem = [stemmer.stem(word) for word in tokens_norm]
    stemming.append(tokens_stem)

    # Tokenisasi akhir
    tokenizing.append(tokens_stem)

# Gabungkan ke DataFrame
tahapan_df = pd.DataFrame({
    'Teks Asli': sample_data,
    'Setelah Stopword Removal': stopwords_removed,
    'Setelah Cleaning': cleaning,
    'Setelah Normalisasi': normalized,
    'Setelah Stemming': stemming,
    'Tokenisasi Akhir': tokenizing
})

# Tentukan lokasi penyimpanan di Google Drive
save_path = "/content/drive/MyDrive/Semester 7/PPW/hasil_preprocessing.csv"

# Simpan otomatis ke Drive
tahapan_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"✅ File berhasil disimpan di {save_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ File berhasil disimpan di /content/drive/MyDrive/Semester 7/PPW/hasil_preprocessing.csv
